# Prueba Técnica – Analista de Datos

**Empresa:** LAX Data Lovers  
**Autor:** Paola Andrea García Tangarife  
**Python:** 3.14  
**Librerías:** pandas, numpy, matplotlib, seaborn, scipy  
**Dataset:** `dataset_prueba_analista.xlsx` (312.200 registros)

## 0. Librerías y estilos para gráficos

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import chi2_contingency, skew, kurtosis
from itertools import combinations
import warnings, os

warnings.filterwarnings("ignore")

OUT = "eda_completo"
os.makedirs(OUT, exist_ok=True)

DARK_BG = "#0F0F14"; CARD_BG = "#1A1A24"; ACCENT  = "#C8FF00"
ACCENT2 = "#7B6FFF"; ACCENT3 = "#FF6B6B"; ACCENT4 = "#FFB86C"
TEXT    = "#E8E8F0"; TEXT_DIM = "#9898AA"; BORDER  = "#2A2A38"
PALETTE = [ACCENT, ACCENT2, ACCENT3, ACCENT4, "#50FA7B",
           "#8BE9FD", "#FF79C6", "#BD93F9", "#F1FA8C"]

def setup_ax(fig, ax, title, subtitle=None):
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(CARD_BG)
    ax.tick_params(colors=TEXT_DIM, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.set_title(title, color=TEXT, fontsize=13, fontweight="bold", pad=14, loc="left")
    if subtitle:
        ax.set_title(subtitle, color=TEXT_DIM, fontsize=9, pad=2, loc="right")
    ax.xaxis.label.set_color(TEXT_DIM); ax.yaxis.label.set_color(TEXT_DIM)
    ax.grid(color=BORDER, linewidth=0.5, alpha=0.6); ax.set_axisbelow(True)

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png", dpi=155, bbox_inches="tight",
                facecolor=DARK_BG, edgecolor="none")
    plt.close(fig)

def fmt_cop(x, _): return f"${x/1e6:.1f}M"

def cramers_v(ct):
    """V de Cramér — mide fuerza de asociación entre variables categóricas."""
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum(); r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

# 1. carga y limpieza de datos

In [37]:
# ── 1.1 Lectura del archivo ───────────────────────────────────────────────────
df_raw = pd.read_excel("dataset_prueba_analista.xlsx")
df_raw.columns = df_raw.columns.str.strip()   # quitar espacios en nombres de columna
print(f"Filas originales  : {len(df_raw):,}")
print(f"Columnas originales: {df_raw.shape[1]}")
print(f"\nColumnas detectadas:\n{df_raw.columns.tolist()}")

Filas originales  : 312,200
Columnas originales: 11

Columnas detectadas:
['ID CLIENTE', 'PDV MAS COMPRADA TRX', 'CONTACTO', 'RANGO COMPRA HISTÓRICA', 'RANGO RECENCIA', 'RFM SEGMENTO', 'CELULAR', 'E_MAIL', 'Ventas', 'HABEAS_DATA', 'CATEGORÍA']


In [38]:
# ── 1.3 Renombrar columnas para análisis ───────────────────────────────────────────────
rename_map = {
    "ID CLIENTE": "id_cliente", "PDV MAS COMPRADA TRX": "pdv",
    "CONTACTO": "canal_contacto", "RANGO COMPRA HISTÓRICA": "rango_compra",
    "RANGO RECENCIA": "rango_recencia", "RFM SEGMENTO": "rfm_segmento",
    "CELULAR": "celular", "E_MAIL": "email", "Ventas": "ventas",
    "HABEAS_DATA": "habeas_data", "CATEGORÍA": "categoria",
}
df = df_raw.rename(columns=rename_map)
print(f"\nColumnas renombradas:\n{df.columns.tolist()}")    



Columnas renombradas:
['id_cliente', 'pdv', 'canal_contacto', 'rango_compra', 'rango_recencia', 'rfm_segmento', 'celular', 'email', 'ventas', 'habeas_data', 'categoria']


In [39]:
# Normalizar texto: strip + upper + convertir string "nan" a NaN real

text_cols = ["pdv", "canal_contacto", "rango_compra", "rango_recencia",
             "rfm_segmento", "habeas_data", "categoria"]

for col in text_cols:
    # Paso 1: convertir todo a string, limpiar espacios y poner mayúsculas
    df[col] = df[col].astype(str).str.strip().str.upper()
    
    # Paso 2: reemplazar TODAS las variantes de vacío por NaN real
    # (no encadenar con lo anterior — aplicar por separado)
    df[col] = df[col].replace({
        "NAN":    np.nan,   # NaN convertido a string
        "NONE":   np.nan,   # None convertido a string
        "":       np.nan,   # cadena vacía
        " ":      np.nan,   # solo espacio
        "N/A":    np.nan,   # variante común
        "NA":     np.nan,   # variante común
        "NULL":   np.nan,   # variante SQL
    })

# ── Verificar resultado ───────────────────────────────────────────────────────
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]

print("Valores nulos por columna después de normalización:")
print(nulos.to_string())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum():,}")

print(f"\nValores únicos por columna de texto:")
for col in text_cols:
    print(f"  {col:<22}: {df[col].nunique():>3} únicos  |  {df[col].isna().sum():>6,} nulos")

# ── Confirmación visual — muestra 5 filas con nulos para verificar ────────────
print(f"\nEjemplo de filas con NaN (primeras 5):")
mask = df[text_cols].isnull().any(axis=1)
print(df[mask][text_cols].head(5).to_string())

Valores nulos por columna después de normalización:
pdv               12680
canal_contacto    15800
rango_compra      12680
rango_recencia     9560
rfm_segmento       9560
celular           22040
email             28280
ventas            18920
habeas_data       25160
categoria         15800

Total de valores nulos: 170,480

Valores únicos por columna de texto:
  pdv                   :   6 únicos  |  12,680 nulos
  canal_contacto        :   6 únicos  |  15,800 nulos
  rango_compra          :   4 únicos  |  12,680 nulos
  rango_recencia        :   8 únicos  |   9,560 nulos
  rfm_segmento          :  13 únicos  |   9,560 nulos
  habeas_data           :   3 únicos  |  25,160 nulos
  categoria             :  19 únicos  |  15,800 nulos

Ejemplo de filas con NaN (primeras 5):
              pdv         canal_contacto                 rango_compra   rango_recencia            rfm_segmento    habeas_data        categoria
0  NU VIVA WAJIRA  E-MAIL, VOZ, WHATSAPP  ENTRE $1.680.001-$2.520.000      7

In [40]:
# ── 1.5 Normalizar columna CATEGORÍA (múltiples variantes de escritura) ───────
# Normalizar CATEGORÍA: 38 variantes → 6 categorías canónicas

def norm_cat(val):
    if pd.isna(val):
        return np.nan

    # Convertir a texto, quitar espacios al inicio/final, pasar a mayúsculas
    # y eliminar TODOS los espacios internos
    v = str(val).strip().upper().replace(" ", "")

    # HIDRATANTE
    if "HIDRATANTE" in v or v.startswith("HIDRAT"):
        return "HIDRATANTE"

    # LIMPIADORES
    if "LIMPIADOR" in v or "LMPIADOR" in v:
        return "LIMPIADORES"

    # SHAMPOO (incluye SHAMPPO y otras variantes)
    if "SHAMPOO" in v or "SHAMPO" in v or "SHAMPPO" in v:
        return "SHAMPOO"

    # ACONDICIONADOR
    if "ACONDICION" in v or "ACONDCION" in v or "ACONDICONADOR" in v:
        return "ACONDICIONADOR"

    # HUMECTANTES
    if "HUMECTANTE" in v or "HUMECTANES" in v:
        return "HUMECTANTES"

    # PROTECTOR SOLAR
    if ("PROTECTOR" in v and "SOLAR" in v) or "PROTECTORSOA" in v:
        return "PROTECTOR SOLAR"

    # Si no coincide con ninguna regla, devolver el valor limpio
    return str(val).strip().upper()


# Aplicar normalización
df["categoria"] = df["categoria"].apply(norm_cat)

# Corregir error tipográfico en habeas_data
df["habeas_data"] = df["habeas_data"].replace({
    "SIN ACTULIZAR": "SIN ACTUALIZAR"
})

# Variables indicadoras
df["tiene_celular"] = df["celular"].notna().astype(int)
df["tiene_email"] = df["email"].notna().astype(int)

# Resumen
N = len(df)
print(f"\nFilas después de limpieza: {N:,}")

print("\nCategorías finales:")
print(sorted(df["categoria"].dropna().unique()))

print("\nValores únicos por columna:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")


Filas después de limpieza: 312,200

Categorías finales:
['ACONDICIONADOR', 'HIDRATANTE', 'HUMECTANTES', 'LIMPIADORES', 'PROTECTOR SOLAR', 'SHAMPOO']

Valores únicos por columna:
  id_cliente: 300200
  pdv: 6
  canal_contacto: 6
  rango_compra: 4
  rango_recencia: 8
  rfm_segmento: 13
  celular: 279403
  email: 267186
  ventas: 262464
  habeas_data: 3
  categoria: 6
  tiene_celular: 2
  tiene_email: 2


In [41]:
# ── 1.6 Crear indicadores binarios de disponibilidad de canal ─────────────────
df["tiene_celular"] = df["celular"].notna().astype(int)
df["tiene_email"]   = df["email"].notna().astype(int)

print(f"\nDataset limpio: {df.shape}")
print(f"Nulos por columna:\n{df.isnull().sum()}")


Dataset limpio: (312200, 13)
Nulos por columna:
id_cliente            0
pdv               12680
canal_contacto    15800
rango_compra      12680
rango_recencia     9560
rfm_segmento       9560
celular           22040
email             28280
ventas            18920
habeas_data       25160
categoria         15800
tiene_celular         0
tiene_email           0
dtype: int64


In [42]:
q1, q3 = df["ventas"].quantile(0.25), df["ventas"].quantile(0.75)
iqr     = q3 - q1
df["outlier_ventas"] = (
    (df["ventas"] < q1 - 1.5*iqr) | (df["ventas"] > q3 + 1.5*iqr)
).astype(int)

def rfm_familia(s):
    if pd.isna(s): return np.nan
    s = str(s)
    for x in ["DIAMANTE", "ORO", "PLATA", "NOVATOS"]:
        if x in s: return x
    return s

def rfm_estado(s):
    if pd.isna(s): return np.nan
    s = str(s)
    if "MUY ACTIVO"    in s: return "MUY ACTIVO"
    if "POR INACTIVAR" in s: return "POR INACTIVAR"
    if "INACTIVO"      in s: return "INACTIVO"
    if "ACTIVO"        in s: return "ACTIVO"
    if "NOVATOS"       in s: return "NOVATOS"
    return s

df["rfm_familia"] = df["rfm_segmento"].apply(rfm_familia)
df["rfm_estado"]  = df["rfm_segmento"].apply(rfm_estado)

df["es_contactable"] = (
    (df["habeas_data"] == "SI") &
    ((df["tiene_celular"] == 1) | (df["tiene_email"] == 1))
).astype(int)

N = len(df)
print(f"Registros totales    : {N:,}")
print(f"Clientes únicos      : {df['id_cliente'].nunique():,}")
print(f"Columnas resultantes : {df.shape[1]}")
print(f"Nulos por columna:\n{df.isnull().sum()}")


Registros totales    : 312,200
Clientes únicos      : 300,200
Columnas resultantes : 17
Nulos por columna:
id_cliente            0
pdv               12680
canal_contacto    15800
rango_compra      12680
rango_recencia     9560
rfm_segmento       9560
celular           22040
email             28280
ventas            18920
habeas_data       25160
categoria         15800
tiene_celular         0
tiene_email           0
outlier_ventas        0
rfm_familia        9560
rfm_estado         9560
es_contactable        0
dtype: int64


In [43]:
# ── ELIMINACIÓN DE DUPLICADOS ─────────────────────────────────────────────────


# Filas antes
filas_antes = len(df)
print(f"Filas antes de eliminar duplicados : {filas_antes:,}")

# Ver cuántos duplicados exactos existen
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas detectadas        : {n_duplicados:,}")

# Mostrar un ejemplo de los duplicados antes de eliminar (opcional)
if n_duplicados > 0:
    ejemplo = df[df.duplicated(keep=False)].sort_values("id_cliente").head(6)
    print(f"\nEjemplo de filas duplicadas:")
    print(ejemplo[["id_cliente", "categoria", "ventas", "rfm_segmento"]].to_string(index=False))

# Eliminar duplicados exactos (mantener primera ocurrencia)
df = df.drop_duplicates(keep="first").reset_index(drop=True)

# Filas después
filas_despues = len(df)
eliminados    = filas_antes - filas_despues

print(f"\nFilas antes            : {filas_antes:,}")
print(f"Filas después          : {filas_despues:,}")
print(f"Duplicados eliminados  : {eliminados:,}  ({eliminados/filas_antes*100:.2f}%)")
print(f"\n✅ df actualizado con {filas_despues:,} filas limpias")

Filas antes de eliminar duplicados : 312,200
Filas duplicadas detectadas        : 3,003

Ejemplo de filas duplicadas:
id_cliente   categoria    ventas        rfm_segmento
CLI-100031 LIMPIADORES   98571.0    PLATA MUY ACTIVO
CLI-100031 LIMPIADORES   98571.0    PLATA MUY ACTIVO
CLI-100154 LIMPIADORES 1808284.0 DIAMANTE MUY ACTIVO
CLI-100154 LIMPIADORES 1808284.0 DIAMANTE MUY ACTIVO
CLI-100192  HIDRATANTE 2087857.0      ORO MUY ACTIVO
CLI-100192  HIDRATANTE 2087857.0      ORO MUY ACTIVO

Filas antes            : 312,200
Filas después          : 309,197
Duplicados eliminados  : 3,003  (0.96%)

✅ df actualizado con 309,197 filas limpias


## Gráfica 01: Calidad de datos 

In [44]:
#  — Calidad de datos · Gráfica 01 ─────────────────────────────────

null_abs     = df.isnull().sum()
null_pct     = null_abs / N * 100
dup_filas    = df.duplicated().sum()
dup_clientes = df["id_cliente"].duplicated().sum()

print(f"Filas duplicadas exactas : {dup_filas:,}")
print(f"ID CLIENTE duplicados    : {dup_clientes:,}  (cada fila = 1 categoría por cliente)")
print(f"\nNulos por columna:")
for col in df.columns:
    if null_abs[col] > 0:
        print(f"  {col:<22}: {null_abs[col]:>7,}  ({null_pct[col]:.1f}%)")

nulos = null_pct[null_pct > 0].sort_values(ascending=True)

# ── Gráfica 01 ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
setup_ax(fig, ax, "Calidad de datos — valores faltantes por variable",
         f"n = {N:,} registros")

bars = ax.barh(nulos.index, nulos.values,
               color=[ACCENT3 if v > 8 else ACCENT2 for v in nulos.values],
               edgecolor="none", height=0.55)

for bar, val in zip(bars, nulos.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", color=TEXT_DIM, fontsize=8.5)

ax.set_xlabel("% de valores nulos", color=TEXT_DIM)
ax.set_xlim(0, nulos.max() + 6)
ax.tick_params(axis="y", labelcolor=TEXT)
fig.tight_layout()
save(fig, "01_calidad_nulos")

Filas duplicadas exactas : 0
ID CLIENTE duplicados    : 8,997  (cada fila = 1 categoría por cliente)

Nulos por columna:
  pdv                   :  12,672  (4.1%)
  canal_contacto        :  15,794  (5.1%)
  rango_compra          :  12,675  (4.1%)
  rango_recencia        :   9,558  (3.1%)
  rfm_segmento          :   9,559  (3.1%)
  celular               :  22,027  (7.1%)
  email                 :  28,249  (9.0%)
  ventas                :  18,912  (6.1%)
  habeas_data           :  25,131  (8.0%)
  categoria             :  15,791  (5.1%)
  rfm_familia           :   9,559  (3.1%)
  rfm_estado            :   9,559  (3.1%)


## Gráfica 02: Estadísticas descriptivas de ventas 

In [45]:
#Estadísticas descriptivas de ventas · Gráficas 02–03
v      = df["ventas"].dropna()
q1, q3 = v.quantile(0.25), v.quantile(0.75)
iqr    = q3 - q1
sk     = skew(v)
ku     = kurtosis(v)
out_low  = v[v < q1 - 1.5*iqr]
out_high = v[v > q3 + 1.5*iqr]
outliers = pd.concat([out_low, out_high])

print(f"Registros con ventas   : {len(v):,}")
print(f"Media                  : ${v.mean():>14,.2f} COP")
print(f"Mediana                : ${v.median():>14,.2f} COP")
print(f"Desv. estándar         : ${v.std():>14,.2f} COP")
print(f"Mínimo                 : ${v.min():>14,.2f} COP")
print(f"Máximo                 : ${v.max():>14,.2f} COP")
print(f"Q1                     : ${q1:>14,.2f} COP")
print(f"Q3                     : ${q3:>14,.2f} COP")
print(f"IQR                    : ${iqr:>14,.2f} COP")
print(f"Sesgo (skewness)       : {sk:>14.4f}")
print(f"Curtosis               : {ku:>14.4f}")
print(f"Outliers IQR           : {len(outliers):>10,}  ({len(outliers)/len(v)*100:.2f}%)")
print(f"  Por debajo fence     : {len(out_low):>10,}")
print(f"  Por encima fence     : {len(out_high):>10,}")
print(f"Fence inferior         : ${q1 - 1.5*iqr:>14,.2f} COP")
print(f"Fence superior         : ${q3 + 1.5*iqr:>14,.2f} COP")


Registros con ventas   : 290,285
Media                  : $  1,550,465.00 COP
Mediana                : $    705,052.00 COP
Desv. estándar         : $  1,803,845.44 COP
Mínimo                 : $     15,005.00 COP
Máximo                 : $  7,999,976.00 COP
Q1                     : $    265,485.00 COP
Q3                     : $  2,196,414.00 COP
IQR                    : $  1,930,929.00 COP
Sesgo (skewness)       :         1.7406
Curtosis               :         2.6108
Outliers IQR           :     21,853  (7.53%)
  Por debajo fence     :          0
  Por encima fence     :     21,853
Fence inferior         : $ -2,630,908.50 COP
Fence superior         : $  5,092,807.50 COP


In [46]:
# ── Gráfica 02: Histograma ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
setup_ax(fig, ax, "Distribución de ventas totales por cliente",
         f"mediana=${v.median()/1e6:.2f}M  |  sesgo={sk:.2f}  |  curtosis={ku:.2f}")
v_filt = v[v <= q3 + 1.5*iqr]
ax.hist(v_filt, bins=70, color=ACCENT, edgecolor=DARK_BG, linewidth=0.3, alpha=0.9)
ax.axvline(v.median(), color=ACCENT3, lw=2, ls="--", label=f"Mediana ${v.median()/1e6:.2f}M")
ax.axvline(v.mean(),   color=ACCENT2, lw=2, ls=":",  label=f"Media   ${v.mean()/1e6:.2f}M")
ax.axvline(q1, color=ACCENT4, lw=1.5, ls="-.", alpha=0.7, label=f"Q1 ${q1/1e6:.2f}M")
ax.axvline(q3, color=ACCENT4, lw=1.5, ls="-.", alpha=0.7, label=f"Q3 ${q3/1e6:.2f}M")
ax.set_xlabel("Ventas (COP)", color=TEXT_DIM)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_cop))
ax.legend(facecolor=CARD_BG, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
fig.tight_layout()
save(fig, "02_distribucion_ventas")

In [47]:
# ── Gráfica 03: Boxplot con outliers ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
setup_ax(fig, ax, "Boxplot de ventas — detección de outliers", "criterio IQR × 1.5")
bp = ax.boxplot(v.values, vert=False, patch_artist=True, widths=0.5,
                medianprops=dict(color=DARK_BG, linewidth=2.5),
                whiskerprops=dict(color=ACCENT2, linewidth=1.5),
                capprops=dict(color=ACCENT2, linewidth=1.5),
                flierprops=dict(marker="o", color=ACCENT3, markersize=2.5,
                                alpha=0.3, linestyle="none"),
                boxprops=dict(linewidth=0))
bp["boxes"][0].set_facecolor(ACCENT2)
bp["boxes"][0].set_alpha(0.6)
ax.set_xlabel("Ventas (COP)", color=TEXT_DIM)
ax.set_yticks([])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_cop))
ax.text(0.01, 0.05,
        f"Outliers: {len(outliers):,} registros ({len(outliers)/len(v)*100:.1f}%)"
        f"  |  Fence superior: ${(q3+1.5*iqr)/1e6:.2f}M",
        transform=ax.transAxes, color=TEXT_DIM, fontsize=8.5)
fig.tight_layout()
save(fig, "03_boxplot_ventas_outliers")

## 4. Variables categóricas · Gráficas 04–11

In [48]:

# ── Gráfica 04: Segmentos RFM ─────────────────────────────────────────────────
rfm_c   = df["rfm_segmento"].value_counts().dropna()
pal_rfm = [ACCENT3 if "INACTIVO" in s or "INACTIVAR" in s
           else ACCENT if "MUY ACTIVO" in s else ACCENT2
           for s in rfm_c.index]
fig, ax = plt.subplots(figsize=(10, 6))
setup_ax(fig, ax, "Distribución de clientes por segmento RFM", "sin registros nulos")
bars = ax.barh(rfm_c.index[::-1], rfm_c.values[::-1],
               color=pal_rfm[::-1], edgecolor="none", height=0.62)
for bar, val in zip(bars, rfm_c.values[::-1]):
    pct = val / rfm_c.sum() * 100
    ax.text(bar.get_width()+300, bar.get_y()+bar.get_height()/2,
            f"{val:,.0f}  ({pct:.1f}%)", va="center", color=TEXT_DIM, fontsize=8.5)
ax.set_xlabel("N° de clientes", color=TEXT_DIM)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
ax.set_xlim(0, rfm_c.max()*1.28)
lg = [mpatches.Patch(color=ACCENT,  label="Muy Activo"),
      mpatches.Patch(color=ACCENT2, label="Activo / Novatos"),
      mpatches.Patch(color=ACCENT3, label="Inactivo / Por inactivar")]
ax.legend(handles=lg, facecolor=CARD_BG, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
fig.tight_layout()
save(fig, "04_rfm_segmentos")



In [49]:
# ── Gráfica 05: Rango de recencia ─────────────────────────────────────────────
order_rec = ["MENOS DE 1 MES","1 A 3 MESES","3 A 6 MESES","6 A 7 MESES",
             "7 A 9 MESES","9 A 12 MESES","12 A 24 MESES","MÁS DE 24 MESES"]
rec_c     = df["rango_recencia"].value_counts().reindex(order_rec).fillna(0)
cmap_rec  = [ACCENT,ACCENT,ACCENT2,ACCENT2,ACCENT3,ACCENT3,ACCENT3,ACCENT3]
fig, ax   = plt.subplots(figsize=(10, 5))
setup_ax(fig, ax, "Distribución por rango de recencia de compra",
         "tiempo desde la última transacción")
bars = ax.bar(range(len(rec_c)), rec_c.values, color=cmap_rec,
              edgecolor=DARK_BG, linewidth=0.4, width=0.66)
for bar, val in zip(bars, rec_c.values):
    pct = val / rec_c.sum() * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+400,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=8)
ax.set_xticks(range(len(rec_c)))
ax.set_xticklabels(rec_c.index, rotation=18, ha="right", color=TEXT_DIM, fontsize=8.5)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
lg2 = [mpatches.Patch(color=ACCENT,  label="Reciente (< 3 m)"),
       mpatches.Patch(color=ACCENT2, label="Intermedio (3-7 m)"),
       mpatches.Patch(color=ACCENT3, label="Inactivo (> 7 m)")]
ax.legend(handles=lg2, facecolor=CARD_BG, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
fig.tight_layout()
save(fig, "05_rango_recencia")

In [50]:
# ── Gráfica 06: Rango compra histórica ────────────────────────────────────────
order_rango = ["MENOS DE $210.000","ENTRE $210.001-$840.000",
               "ENTRE $1.680.001-$2.520.000","MAS DE $2.520.000"]
rc_c = df["rango_compra"].value_counts().reindex(order_rango).fillna(0)
fig, ax = plt.subplots(figsize=(9, 5))
setup_ax(fig, ax, "Distribución por rango de compra histórica",
         "clientes con dato disponible")
bars = ax.bar(range(len(rc_c)), rc_c.values,
              color=[ACCENT2, ACCENT, ACCENT, ACCENT3],
              edgecolor=DARK_BG, linewidth=0.5, width=0.6)
for bar, val in zip(bars, rc_c.values):
    pct = val / rc_c.sum() * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1200,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=8.5)
ax.set_xticks(range(len(rc_c)))
ax.set_xticklabels([r.replace("ENTRE ","") for r in rc_c.index],
                   rotation=12, ha="right", color=TEXT_DIM, fontsize=8.5)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
fig.tight_layout()
save(fig, "06_rango_compra_historica")

In [51]:
# ── Gráfica 07: PDV ───────────────────────────────────────────────────────────
pdv_c = df["pdv"].value_counts().dropna()
fig, ax = plt.subplots(figsize=(9, 5))
setup_ax(fig, ax, "Clientes por punto de venta principal",
         "PDV = tienda con mayor n° de transacciones del cliente")
bars = ax.bar(range(len(pdv_c)), pdv_c.values,
              color=PALETTE[:len(pdv_c)], edgecolor=DARK_BG, linewidth=0.4, width=0.6)
for bar, val in zip(bars, pdv_c.values):
    pct = val / pdv_c.sum() * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+150,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=9)
ax.set_xticks(range(len(pdv_c)))
ax.set_xticklabels([p.replace("NU ","") for p in pdv_c.index],
                   rotation=15, ha="right", color=TEXT_DIM, fontsize=9)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
fig.tight_layout()
save(fig, "07_punto_de_venta")

In [52]:
# ── Gráfica 08: Categorías ────────────────────────────────────────────────────
cat_c = df["categoria"].value_counts().dropna()
fig, ax = plt.subplots(figsize=(9, 5))
setup_ax(fig, ax, "Distribución de clientes por categoría de producto",
         "después de normalización de variantes")
bars = ax.bar(range(len(cat_c)), cat_c.values,
              color=PALETTE[:len(cat_c)], edgecolor=DARK_BG, linewidth=0.4, width=0.6)
for bar, val in zip(bars, cat_c.values):
    pct = val / cat_c.sum() * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=8.5)
ax.set_xticks(range(len(cat_c)))
ax.set_xticklabels(cat_c.index, rotation=18, ha="right", color=TEXT_DIM, fontsize=9)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
fig.tight_layout()
save(fig, "08_categorias_producto")

In [53]:
# ── Gráfica 09: Canal de contacto ─────────────────────────────────────────────
con_c = df["canal_contacto"].value_counts().dropna()
fig, ax = plt.subplots(figsize=(9, 5))
setup_ax(fig, ax, "Distribución de clientes por canal de contacto", "excluye nulos")
bars = ax.bar(range(len(con_c)), con_c.values,
              color=PALETTE[:len(con_c)], edgecolor=DARK_BG, linewidth=0.4, width=0.65)
for bar, val in zip(bars, con_c.values):
    pct = val / con_c.sum() * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=8)
ax.set_xticks(range(len(con_c)))
ax.set_xticklabels(con_c.index, rotation=22, ha="right", color=TEXT_DIM, fontsize=8.5)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
fig.tight_layout()
save(fig, "09_canal_contacto")

In [54]:
# ── Gráfica 10: Habeas Data ───────────────────────────────────────────────────
hd = df["habeas_data"].value_counts().dropna()
fig, ax = plt.subplots(figsize=(8, 5))
setup_ax(fig, ax, "Autorización Habeas Data", "clientes con dato disponible")
bars = ax.bar(hd.index, hd.values,
              color=[ACCENT, ACCENT3, ACCENT2], edgecolor=DARK_BG, width=0.5)
for bar, val in zip(bars, hd.values):
    pct = val / hd.sum() * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+800,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=9)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
fig.tight_layout()
save(fig, "10_habeas_data")


In [55]:
# ── Gráfica 11: Disponibilidad de canales ─────────────────────────────────────
canal_df = pd.DataFrame({
    "Canal":    ["Celular", "Email"],
    "Con dato": [df["tiene_celular"].sum(), df["tiene_email"].sum()],
    "Sin dato": [N - df["tiene_celular"].sum(), N - df["tiene_email"].sum()],
})
fig, ax = plt.subplots(figsize=(8, 5))
setup_ax(fig, ax, "Disponibilidad de canales de contacto", "celular vs email")
x, w = np.arange(2), 0.35
ax.bar(x-w/2, canal_df["Con dato"], w, color=ACCENT,  edgecolor=DARK_BG, label="Con dato")
ax.bar(x+w/2, canal_df["Sin dato"], w, color=ACCENT3, edgecolor=DARK_BG, label="Sin dato")
for i, row in canal_df.iterrows():
    pct = row["Con dato"] / (row["Con dato"] + row["Sin dato"]) * 100
    ax.text(i-w/2, row["Con dato"]+1500, f"{pct:.1f}%",
            ha="center", color=TEXT_DIM, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(canal_df["Canal"], color=TEXT_DIM, fontsize=11)
ax.set_ylabel("N° de clientes", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
ax.legend(facecolor=CARD_BG, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
fig.tight_layout()
save(fig, "11_disponibilidad_canales")

## 5. Ventas cruzadas · Gráficas 12–14

In [56]:

# Estadísticas ventas por RFM
rfm_stats = (df.groupby("rfm_segmento")["ventas"]
               .agg(["mean","median","std","count"])
               .rename(columns={"mean":"Media","median":"Mediana",
                                "std":"Desv.Est","count":"N"})
               .sort_values("Mediana", ascending=False)
               .round(0))
print("Ventas por Segmento RFM:")
print(rfm_stats.to_string())

# Estadísticas ventas por Categoría
cat_stats = (df.groupby("categoria")["ventas"]
               .agg(["mean","median","std","count"])
               .rename(columns={"mean":"Media","median":"Mediana",
                                "std":"Desv.Est","count":"N"})
               .sort_values("Mediana", ascending=False)
               .round(0))
print("\nVentas por Categoría:")
print(cat_stats.to_string())


Ventas por Segmento RFM:
                            Media    Mediana   Desv.Est      N
rfm_segmento                                                  
DIAMANTE INACTIVO       3686432.0  2555072.0  1945931.0  37077
DIAMANTE ACTIVO         3688231.0  2549423.0  1947257.0  17629
DIAMANTE POR INACTIVAR  3669555.0  2547090.0  1936362.0  10984
DIAMANTE MUY ACTIVO     3683789.0  2535733.0  1944379.0  13097
ORO ACTIVO              1308537.0  1680334.0   814479.0  17796
ORO POR INACTIVAR       1313154.0  1680046.0   816667.0  10964
ORO INACTIVO            1308381.0   837249.0   817185.0  37647
ORO MUY ACTIVO          1297115.0   828925.0   816334.0  13038
NOVATOS                  388831.0   222339.0   613270.0  43930
PLATA INACTIVO           319017.0   212958.0   246356.0  37521
PLATA MUY ACTIVO         316895.0   207911.0   246215.0  12875
PLATA ACTIVO             315374.0   207284.0   244553.0  17739
PLATA POR INACTIVAR      312816.0   205697.0   244042.0  11203

Ventas por Categoría:
       

In [57]:
# ── Gráfica 12: Boxplot ventas × RFM ─────────────────────────────────────────
rfm_v     = df[df["ventas"].notna() & df["rfm_segmento"].notna()].copy()
order_rfm = rfm_v.groupby("rfm_segmento")["ventas"].median().sort_values().index.tolist()
data_box  = [rfm_v[rfm_v["rfm_segmento"]==s]["ventas"].values for s in order_rfm]
fig, ax   = plt.subplots(figsize=(12, 6))
setup_ax(fig, ax, "Ventas por segmento RFM",
         "mediana y dispersión — sin outliers extremos")
bp = ax.boxplot(data_box, vert=False, patch_artist=True, showfliers=False,
                medianprops=dict(color=DARK_BG, linewidth=2.5),
                whiskerprops=dict(color=BORDER), capprops=dict(color=BORDER),
                boxprops=dict(linewidth=0))
for patch, seg in zip(bp["boxes"], order_rfm):
    c = (ACCENT3 if "INACTIVO" in seg or "INACTIVAR" in seg
         else ACCENT if "MUY ACTIVO" in seg else ACCENT2)
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax.set_yticks(range(1, len(order_rfm)+1))
ax.set_yticklabels(order_rfm, color=TEXT_DIM, fontsize=8.5)
ax.set_xlabel("Ventas (COP)", color=TEXT_DIM)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_cop))
lg = [mpatches.Patch(color=ACCENT,  label="Muy Activo"),
      mpatches.Patch(color=ACCENT2, label="Activo / Novatos"),
      mpatches.Patch(color=ACCENT3, label="Inactivo / Por inactivar")]
ax.legend(handles=lg, facecolor=CARD_BG, edgecolor=BORDER, labelcolor=TEXT,
          fontsize=9, loc="lower right")
fig.tight_layout()
save(fig, "12_ventas_por_rfm_boxplot")

In [58]:
# ── Gráfica 13: Venta mediana por categoría ───────────────────────────────────
cv = (df.groupby("categoria")["ventas"]
        .agg(["median","count"]).dropna().sort_values("median"))
fig, ax = plt.subplots(figsize=(9, 5))
setup_ax(fig, ax, "Venta mediana por categoría de producto",
         "clientes con dato de venta disponible")
bars = ax.barh(cv.index, cv["median"],
               color=PALETTE[:len(cv)], edgecolor="none", height=0.55)
for bar, val in zip(bars, cv["median"]):
    ax.text(bar.get_width()+8000, bar.get_y()+bar.get_height()/2,
            f"${val/1e6:.2f}M", va="center", color=TEXT_DIM, fontsize=8.5)
ax.set_xlabel("Venta mediana (COP)", color=TEXT_DIM)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_cop))
ax.set_xlim(0, cv["median"].max()*1.22)
fig.tight_layout()
save(fig, "13_ventas_por_categoria")



In [59]:
# ── Gráfica 14: Boxplot ventas × PDV ─────────────────────────────────────────
pdv_v     = df[df["ventas"].notna() & df["pdv"].notna()].copy()
order_pdv = pdv_v.groupby("pdv")["ventas"].median().sort_values().index.tolist()
data_pdv  = [pdv_v[pdv_v["pdv"]==p]["ventas"].values for p in order_pdv]
fig, ax   = plt.subplots(figsize=(10, 5))
setup_ax(fig, ax, "Ventas por punto de venta", "sin valores atípicos extremos")
bp2 = ax.boxplot(data_pdv, vert=False, patch_artist=True, showfliers=False,
                 medianprops=dict(color=DARK_BG, linewidth=2.5),
                 whiskerprops=dict(color=BORDER), capprops=dict(color=BORDER),
                 boxprops=dict(linewidth=0))
for patch, col in zip(bp2["boxes"], PALETTE):
    patch.set_facecolor(col); patch.set_alpha(0.75)
ax.set_yticks(range(1, len(order_pdv)+1))
ax.set_yticklabels([p.replace("NU ","") for p in order_pdv], color=TEXT_DIM, fontsize=9)
ax.set_xlabel("Ventas (COP)", color=TEXT_DIM)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_cop))
fig.tight_layout()
save(fig, "14_ventas_por_pdv_boxplot")

# 6. Correlación entre categorías · Gráficas 15–18

In [60]:
# ── Bloque 6 — Correlación entre categorías · Gráficas 15–18 ─────────────────

CATS = ["HIDRATANTE", "LIMPIADORES", "SHAMPOO",
        "ACONDICIONADOR", "HUMECTANTES", "PROTECTOR SOLAR"]

# Crear dummies: 1 si el cliente compró esa categoría, 0 si no
for cat in CATS:
    df[cat] = (df["categoria"] == cat).astype(int)

# ── Tabla chi² y V de Cramér para los 15 pares posibles ──────────────────────
print(f"{'Par':<40} {'Chi²':>12} {'p-valor':>12} {'V Cramér':>10} {'Sig.':>6}")
print("-" * 82)

resultados = []
for c1, c2 in combinations(CATS, 2):
    ct               = pd.crosstab(df[c1], df[c2])
    chi2_val, p_val, dof, _ = chi2_contingency(ct)
    v_val            = cramers_v(ct)
    sig              = ("***" if p_val < 0.001
                        else "**"  if p_val < 0.01
                        else "*"   if p_val < 0.05
                        else "ns")
    fuerza           = ("MUY ALTA"  if v_val >= 0.35
                        else "ALTA"     if v_val >= 0.25
                        else "MODERADA" if v_val >= 0.15
                        else "BAJA"     if v_val >= 0.05
                        else "MUY BAJA")
    resultados.append({
        "par":      f"{c1} × {c2}",
        "c1":       c1,
        "c2":       c2,
        "chi2":     chi2_val,
        "p_valor":  p_val,
        "v_cramer": v_val,
        "sig":      sig,
        "fuerza":   fuerza,
    })
    print(f"{c1+' × '+c2:<40} {chi2_val:>12,.1f} {p_val:>12.2e} {v_val:>10.4f} {sig:>6}")

res_df = pd.DataFrame(resultados).sort_values("v_cramer", ascending=False)

print(f"\nPar más asociado  : {res_df.iloc[0]['par']}"
      f"  (V={res_df.iloc[0]['v_cramer']:.4f}  {res_df.iloc[0]['fuerza']})")
print(f"Par menos asociado: {res_df.iloc[-1]['par']}"
      f"  (V={res_df.iloc[-1]['v_cramer']:.4f}  {res_df.iloc[-1]['fuerza']})")

Par                                              Chi²      p-valor   V Cramér   Sig.
----------------------------------------------------------------------------------
HIDRATANTE × LIMPIADORES                     26,364.7     0.00e+00     0.2920    ***
HIDRATANTE × SHAMPOO                         12,852.1     0.00e+00     0.2039    ***
HIDRATANTE × ACONDICIONADOR                  12,841.5     0.00e+00     0.2038    ***
HIDRATANTE × HUMECTANTES                     12,822.8     0.00e+00     0.2036    ***
HIDRATANTE × PROTECTOR SOLAR                 12,762.2     0.00e+00     0.2032    ***
LIMPIADORES × SHAMPOO                        12,824.4     0.00e+00     0.2037    ***
LIMPIADORES × ACONDICIONADOR                 12,813.7     0.00e+00     0.2036    ***
LIMPIADORES × HUMECTANTES                    12,795.1     0.00e+00     0.2034    ***
LIMPIADORES × PROTECTOR SOLAR                12,734.6     0.00e+00     0.2029    ***
SHAMPOO × ACONDICIONADOR                      6,246.1     0.00e+00 

In [61]:
# ── Gráfica 15: Heatmap V de Cramér ──────────────────────────────────────────
cramer_mat = pd.DataFrame(np.nan, index=CATS, columns=CATS)
for _, row in res_df.iterrows():
    cramer_mat.loc[row["c1"], row["c2"]] = row["v_cramer"]
    cramer_mat.loc[row["c2"], row["c1"]] = row["v_cramer"]
cramer_arr = cramer_mat.values.copy().astype(float)
np.fill_diagonal(cramer_arr, 1.0)
cramer_mat = pd.DataFrame(cramer_arr, index=CATS, columns=CATS)

mask = np.zeros_like(cramer_mat, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(CARD_BG)
sns.heatmap(cramer_mat, ax=ax, mask=mask, cmap="YlGn", vmin=0, vmax=1,
            linewidths=0.5, linecolor=DARK_BG,
            annot=True, fmt=".3f", annot_kws={"size": 10, "color": "#111"},
            cbar_kws={"shrink": 0.7})
ax.set_title("Fuerza de asociación entre categorías — V de Cramér",
             color=TEXT, fontsize=13, fontweight="bold", pad=14, loc="left")
ax.set_title("0=sin relación  |  1=relación perfecta",
             color=TEXT_DIM, fontsize=9, pad=2, loc="right")
ax.tick_params(axis="x", colors=TEXT_DIM, rotation=30, labelsize=9)
ax.tick_params(axis="y", colors=TEXT_DIM, rotation=0,  labelsize=9)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT_DIM)
fig.tight_layout()
save(fig, "15_cramer_v_heatmap")

In [62]:
# ── Gráfica 16: Ranking de asociaciones ──────────────────────────────────────
top = res_df.sort_values("v_cramer", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
setup_ax(fig, ax, "Ranking de asociación entre pares de categorías",
         "V de Cramér — mayor valor = mayor asociación")
cols_bar = [ACCENT if v >= 0.25 else ACCENT2 if v >= 0.15 else ACCENT3
            for v in top["v_cramer"]]
bars = ax.barh(top["par"], top["v_cramer"],
               color=cols_bar, edgecolor="none", height=0.6)
for bar, val, fuerza in zip(bars, top["v_cramer"], top["fuerza"]):
    ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
            f"{val:.4f}  [{fuerza}]", va="center", color=TEXT_DIM, fontsize=8.5)
ax.set_xlabel("V de Cramér", color=TEXT_DIM)
ax.set_xlim(0, top["v_cramer"].max()*1.35)
ax.axvline(0.15, color=ACCENT4, lw=1, ls="--", alpha=0.6, label="Umbral moderado (0.15)")
ax.axvline(0.25, color=ACCENT,  lw=1, ls="--", alpha=0.6, label="Umbral alto (0.25)")
ax.legend(facecolor=CARD_BG, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
fig.tight_layout()
save(fig, "16_ranking_asociaciones_categorias")

In [63]:
# ── Gráfica 17: Co-ocurrencia absoluta ───────────────────────────────────────
cli_cats = (df.dropna(subset=["id_cliente","categoria"])
              .groupby("id_cliente")["categoria"].apply(list))
comat = pd.DataFrame(0, index=CATS, columns=CATS)
for cats in cli_cats:
    cu = list(set(cats) & set(CATS))
    for i, c1 in enumerate(cu):
        for c2 in cu[i:]:
            comat.loc[c1, c2] += 1
            if c1 != c2: comat.loc[c2, c1] += 1
arr = comat.values.copy().astype(float)
np.fill_diagonal(arr, 0)
comat = pd.DataFrame(arr, index=CATS, columns=CATS)

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(CARD_BG)
sns.heatmap(comat, ax=ax, cmap="Blues", linewidths=0.5, linecolor=DARK_BG,
            annot=True, fmt=".0f", annot_kws={"size": 9.5},
            cbar_kws={"shrink": 0.7})
ax.set_title("Co-ocurrencia de categorías en historial por cliente",
             color=TEXT, fontsize=13, fontweight="bold", pad=14, loc="left")
ax.tick_params(axis="x", colors=TEXT_DIM, rotation=30, labelsize=9)
ax.tick_params(axis="y", colors=TEXT_DIM, rotation=0,  labelsize=9)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT_DIM)
fig.tight_layout()
save(fig, "17_coocurrencia_categorias")

In [64]:
# ── Gráfica 18: Heatmap categoría × RFM ──────────────────────────────────────
pivot      = (df.dropna(subset=["categoria","rfm_segmento"])
                .groupby(["categoria","rfm_segmento"]).size().unstack(fill_value=0))
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)
fig, ax    = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(CARD_BG)
sns.heatmap(pivot_norm, ax=ax, cmap="YlGn", linewidths=0.4, linecolor=DARK_BG,
            annot=True, fmt=".1%", annot_kws={"size": 7.5, "color": "#111"},
            cbar_kws={"shrink": 0.7})
ax.set_title("Composición de segmento RFM dentro de cada categoría",
             color=TEXT, fontsize=13, fontweight="bold", pad=14, loc="left")
ax.tick_params(axis="x", colors=TEXT_DIM, rotation=30, labelsize=8.5)
ax.tick_params(axis="y", colors=TEXT_DIM, rotation=0,  labelsize=9)
ax.set_xlabel("Segmento RFM", color=TEXT_DIM)
ax.set_ylabel("Categoría", color=TEXT_DIM)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT_DIM)
fig.tight_layout()
save(fig, "18_heatmap_categoria_rfm")


# 8. Insights accionables · Gráficas 19–22

In [65]:

# ── A. Alcance real de marketing ──────────────────────────────────────────────
total_hd_si = (df["habeas_data"] == "SI").sum()
hd_no       = (df["habeas_data"] == "NO").sum()
hd_sin_act  = (df["habeas_data"] == "SIN ACTUALIZAR").sum()
con_celular = df[(df["habeas_data"] == "SI") & (df["tiene_celular"] == 1)].shape[0]
con_email   = df[(df["habeas_data"] == "SI") & (df["tiene_email"]   == 1)].shape[0]
sin_canal   = df[(df["habeas_data"] == "SI") & (df["tiene_celular"] == 0)
                 & (df["tiene_email"] == 0)].shape[0]

print("A. ALCANCE REAL DE MARKETING")
print(f"   Con Habeas Data SI           : {total_hd_si:>10,}  ({total_hd_si/N*100:.1f}%)")
print(f"   Con HD-SI + Celular          : {con_celular:>10,}  ({con_celular/N*100:.1f}%)")
print(f"   Con HD-SI + Email            : {con_email:>10,}  ({con_email/N*100:.1f}%)")
print(f"   Con HD-SI sin ningún canal   : {sin_canal:>10,}  ({sin_canal/N*100:.1f}%)")
print(f"   Habeas Data = NO             : {hd_no:>10,}  ({hd_no/N*100:.1f}%)")
print(f"   Habeas Data = SIN ACTUALIZAR : {hd_sin_act:>10,}  ({hd_sin_act/N*100:.1f}%)")

# ── B. Segmentos inactivos ────────────────────────────────────────────────────
inactivos   = df[df["rfm_segmento"].str.contains("INACTIVO|INACTIVAR", na=False)]
inact_stats = inactivos.groupby("rfm_segmento")["ventas"].agg(["count", "median"]).round(0)
total_inact = inactivos["id_cliente"].nunique()

print(f"\nB. SEGMENTOS INACTIVOS")
print(inact_stats.to_string())
print(f"   Total clientes inactivos: {total_inact:,}")

# ── C. Top pares cross-sell ───────────────────────────────────────────────────
comat_flat = (comat.stack()
                   .reset_index()
                   .rename(columns={"level_0": "cat_a", "level_1": "cat_b", 0: "n"})
                   .query("cat_a < cat_b")
                   .sort_values("n", ascending=False)
                   .head(6))

print("\nC. TOP PARES CROSS-SELL")
for _, row in comat_flat.iterrows():
    print(f"   {row['cat_a']:<22} + {row['cat_b']:<22} → {row['n']:,.0f} clientes en común")

# ── D. Novatos ────────────────────────────────────────────────────────────────
novatos = df[df["rfm_segmento"] == "NOVATOS"]

print(f"\nD. NOVATOS")
print(f"   Total                   : {len(novatos):,}")
print(f"   Venta mediana           : ${novatos['ventas'].median():,.0f} COP")
print(f"   Categoría más frecuente : {novatos['categoria'].value_counts().index[0]}")
print(f"   % con celular           : {novatos['tiene_celular'].mean()*100:.1f}%")
print(f"   % con email             : {novatos['tiene_email'].mean()*100:.1f}%")

A. ALCANCE REAL DE MARKETING
   Con Habeas Data SI           :    170,288  (54.5%)
   Con HD-SI + Celular          :    158,337  (50.7%)
   Con HD-SI + Email            :    154,710  (49.6%)
   Con HD-SI sin ningún canal   :      1,080  (0.3%)
   Habeas Data = NO             :     42,666  (13.7%)
   Habeas Data = SIN ACTUALIZAR :     71,112  (22.8%)

B. SEGMENTOS INACTIVOS
                        count     median
rfm_segmento                            
DIAMANTE INACTIVO       37077  2555072.0
DIAMANTE POR INACTIVAR  10984  2547090.0
ORO INACTIVO            37647   837249.0
ORO POR INACTIVAR       10964  1680046.0
PLATA INACTIVO          37521   212958.0
PLATA POR INACTIVAR     11203   205697.0
   Total clientes inactivos: 150,470

C. TOP PARES CROSS-SELL
   HIDRATANTE             + SHAMPOO                → 48 clientes en común
   HIDRATANTE             + HUMECTANTES            → 48 clientes en común
   LIMPIADORES            + SHAMPOO                → 46 clientes en común
   HIDRATANT

In [66]:
# ── Gráfica 19: Alcance marketing ─────────────────────────────────────────────
labels    = ["HD=SI\n+Celular","HD=SI\n+Email","HD=SI\nSin canal",
             "HD=NO","HD=Sin\nActualizar","Dato\nnulo"]
values    = [con_celular, con_email, sin_canal, hd_no, hd_sin_act,
             int(df["habeas_data"].isna().sum())]
colors_hd = [ACCENT, ACCENT2, ACCENT4, ACCENT3, "#9898AA", BORDER]
fig, ax   = plt.subplots(figsize=(10, 5))
setup_ax(fig, ax, "Alcance real de marketing — Habeas Data y canales",
         "base contactable vs no contactable")
bars = ax.bar(range(len(labels)), values, color=colors_hd,
              edgecolor=DARK_BG, linewidth=0.4, width=0.65)
for bar, val in zip(bars, values):
    pct = val / N * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
            f"{val:,.0f}\n({pct:.1f}%)", ha="center", va="bottom",
            color=TEXT_DIM, fontsize=8.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, color=TEXT_DIM, fontsize=9)
ax.set_ylabel("N° de registros", color=TEXT_DIM)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
fig.tight_layout()
save(fig, "19_alcance_marketing")



In [67]:
# ── Gráfica 20: RFM × Categoría — venta mediana ──────────────────────────────
rfm_cat_med = (df.groupby(["rfm_segmento","categoria"])["ventas"]
                 .median().unstack(fill_value=np.nan))
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(CARD_BG)
sns.heatmap(rfm_cat_med/1e6, ax=ax, cmap="YlGn",
            linewidths=0.4, linecolor=DARK_BG,
            annot=True, fmt=".1f", annot_kws={"size": 8, "color": "#111"},
            cbar_kws={"shrink": 0.6})
ax.set_title("Venta mediana (M COP) por segmento RFM y categoría",
             color=TEXT, fontsize=13, fontweight="bold", pad=14, loc="left")
ax.tick_params(axis="x", colors=TEXT_DIM, rotation=30, labelsize=8.5)
ax.tick_params(axis="y", colors=TEXT_DIM, rotation=0,  labelsize=8.5)
ax.set_xlabel("Categoría", color=TEXT_DIM)
ax.set_ylabel("Segmento RFM", color=TEXT_DIM)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(colors=TEXT_DIM)
cbar.set_label("Venta mediana (M COP)", color=TEXT_DIM)
fig.tight_layout()
save(fig, "20_rfm_categoria_ventas_heatmap")



In [68]:
# ── Gráfica 21: Top cross-sell ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
setup_ax(fig, ax, "Top 6 pares con mayor potencial de cross-sell",
         "clientes que compraron ambas categorías en su historial")
bars = ax.barh(
    [f"{r['cat_a']}\n+ {r['cat_b']}" for _, r in comat_flat.iterrows()],
    comat_flat["n"].values,
    color=PALETTE[:6], edgecolor="none", height=0.6
)
for bar, val in zip(bars, comat_flat["n"].values):
    ax.text(bar.get_width()+200, bar.get_y()+bar.get_height()/2,
            f"{val:,.0f} clientes", va="center", color=TEXT_DIM, fontsize=9)
ax.set_xlabel("N° de clientes que compraron ambas categorías", color=TEXT_DIM)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
ax.set_xlim(0, comat_flat["n"].max()*1.22)
fig.tight_layout()
save(fig, "21_top_crosssell_pares")



In [69]:
# ── Gráfica 22: Recencia × RFM ───────────────────────────────────────────────
order_rec2 = ["MENOS DE 1 MES","1 A 3 MESES","3 A 6 MESES","6 A 7 MESES",
              "7 A 9 MESES","9 A 12 MESES","12 A 24 MESES","MÁS DE 24 MESES"]
rec_rfm = (df.dropna(subset=["rango_recencia","rfm_segmento"])
             .groupby(["rango_recencia","rfm_segmento"])
             .size().unstack(fill_value=0))
rec_rfm = rec_rfm.reindex([r for r in order_rec2 if r in rec_rfm.index])
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(CARD_BG)
sns.heatmap(rec_rfm, ax=ax, cmap="RdYlGn_r",
            linewidths=0.4, linecolor=DARK_BG,
            annot=True, fmt=",", annot_kws={"size": 7.5, "color": "#111"},
            cbar_kws={"shrink": 0.6})
ax.set_title("Concentración de clientes por recencia y segmento RFM",
             color=TEXT, fontsize=13, fontweight="bold", pad=14, loc="left")
ax.tick_params(axis="x", colors=TEXT_DIM, rotation=30, labelsize=8.5)
ax.tick_params(axis="y", colors=TEXT_DIM, rotation=0,  labelsize=8.5)
ax.set_xlabel("Segmento RFM", color=TEXT_DIM)
ax.set_ylabel("Rango de recencia", color=TEXT_DIM)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT_DIM)
fig.tight_layout()
save(fig, "22_recencia_rfm_heatmap")